# Session 11 - final metrics, report, human study


**CPU, minutes.** Merges the outputs of every earlier session into the final tables,
figures and report.

Attach **every** output dataset: `cca-s1-parsed`, `cca-s2-controls`, `cca-s4-cnn`,
every `cca-s5-vlm-*`, `cca-s6-extras`, `cca-s7-proxy`, `cca-s8-attr`,
`cca-s10a-cset`, `cca-s10b-cset`. Overlapping runs are de-duplicated by content key,
so attaching more than needed is safe; attaching too few silently drops rows.

In [ ]:
SESSION = "S11 final"

# ============================== CONFIG ==============================
SPLIT        = "test"    # every reported number comes from TEST
TAU          = 0.5
TAUS         = "0.3,0.5,0.7"
N_BOOT       = 2000
SEED         = 0
HUMAN_STUDY_N = 100
HUMAN_DETECTOR = "qwen25vl:Qwen/Qwen2.5-VL-3B-Instruct"  # the main VLM run of Session 5
ANNOTATORS   = 3

In [ ]:
# ---------------------------------------------------------------- BOOT
# Locates the code dataset wherever it is mounted, puts it on sys.path,
# prints the attached inputs, and records provenance.  The search is by
# file name, so the dataset's mount name does not matter.
import os, sys, subprocess, json, time

def _find_code():
    for root in ("/kaggle/input", "."):
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, files in os.walk(root):
            dirnames[:] = [d for d in dirnames if not d.startswith(".")]
            if os.path.basename(dirpath) == "ccaudit" and "kaggle_utils.py" in files:
                return os.path.dirname(dirpath)
    raise FileNotFoundError(
        "Could not find the ccaudit package.\n"
        "Add Input -> your code dataset (<your-code-dataset>), and check that "
        "the preview shows ccaudit/kaggle_utils.py at the top level.")

CODE = _find_code()
if CODE not in sys.path:
    sys.path.insert(0, CODE)
# Child processes do not inherit sys.path.  Every `python -m ccaudit.<module>`
# below runs as a subprocess, so the code directory must be on PYTHONPATH.
os.environ["PYTHONPATH"] = CODE + os.pathsep + os.environ.get("PYTHONPATH", "")
from ccaudit import kaggle_utils as KU
from ccaudit import common as C

OUT = KU.work_dir("audit")
TMP = KU.temp_dir()
os.environ["HF_HOME"] = KU.temp_dir("hf")          # model weights stay out of /kaggle/working
os.environ["TOKENIZERS_PARALLELISM"] = "false"
KU.session_header(SESSION, OUT)
print("code:", CODE)

In [ ]:
# ------------------------------------------------------- SELF TEST (always)
# The self-test suite runs in under a minute and needs no dataset.  Each check
# corresponds to a failure mode that would produce plausible-looking but
# incorrect numbers, so a failure here invalidates everything that follows.
rc = KU.sh(f"{sys.executable} {CODE}/scripts/selftest.py", check=False)
if rc != 0:
    raise SystemExit("SELF TEST FAILED -- inspect the failures above before proceeding.")

In [ ]:
# ------------------------------------------------- find everything
INDEX = KU.find_parsed_index()
if not INDEX:
    raise SystemExit("Add Input -> `cca-s1-parsed`.")
recs, meta = C.load_index(INDEX)
VOCAB = meta.get("vocab", "face8")

RUN_DIRS = KU.find_run_dirs()
print(f"INDEX = {INDEX}\n{len(recs)} samples, vocab={VOCAB}\n")
print("run directories found:")
for d in RUN_DIRS:
    print("  ", d)
if not RUN_DIRS:
    raise SystemExit("No raw_*.json found. Attach the session outputs.")
RAW = ",".join(RUN_DIRS)

In [ ]:
# ------------------------------------------------- localization on TEST
KU.sh(f'{sys.executable} -m ccaudit.m10_localization --index "{INDEX}" '
      f'--out "{OUT}/loc" --split {SPLIT}', check=False,
      log=f"{OUT}/logs/m10.log")
LOC = f"{OUT}/loc/localization.json"

In [ ]:
# ------------------------------------------------- metrics
common = (f'--raw "{RAW}" --out "{OUT}/metrics" --split {SPLIT} --tau {TAU} '
          f'--taus {TAUS} --n-boot {N_BOOT} --seed {SEED} --vocab {VOCAB}')
KU.sh(f'{sys.executable} -m ccaudit.m6_metrics {common} '
      f'--localization "{LOC}" --by method', check=False,
      log=f"{OUT}/logs/m6.log")
KU.sh(f'{sys.executable} -m ccaudit.m6_metrics {common} --coarse',
      check=False, log=f"{OUT}/logs/m6.log")

In [ ]:
# ------------------------------------------------- side analyses
KU.sh(f'{sys.executable} -m ccaudit.m11_proxy --raw "{RAW}" '
      f'--out "{OUT}/proxy" --split {SPLIT} --vocab {VOCAB}', check=False)
KU.sh(f'{sys.executable} -m ccaudit.m12_text_regions --raw "{RAW}" '
      f'--out "{OUT}/text" --vocab {VOCAB}', check=False)
KU.sh(f'{sys.executable} -m ccaudit.m14_attribution --raw "{RAW}" '
      f'--out "{OUT}/attr" --vocab {VOCAB}', check=False)

In [ ]:
# ------------------------------------------------- report + csv
KU.sh(f'{sys.executable} -m ccaudit.m7_report '
      f'--metrics "{OUT}/metrics/metrics.json" '
      f'--coarse "{OUT}/metrics/metrics_coarse.json" '
      f'--by-method "{OUT}/metrics/metrics_by_method.json" '
      f'--raw "{RAW}" --out "{OUT}/report.html" '
      f'--csv "{OUT}/results_table.csv" --figures-dir "{OUT}/figures" '
      f'--title "Counterfactual citation audit - final report ({SPLIT})"', check=False)

In [ ]:
# ------------------------------------------------- human study materials
# HUMAN_DETECTOR selects the run whose items are shown to annotators; it is
# set explicitly in the config cell so that the study is tied to a named run.
det = f' --detector "{HUMAN_DETECTOR}"' if HUMAN_DETECTOR else ""
KU.sh(f'{sys.executable} -m ccaudit.m8_human_study --index "{INDEX}" '
      f'--raw "{RAW}" --out "{OUT}/human_study" --n-items {HUMAN_STUDY_N} '
      f'--split {SPLIT} --annotators {ANNOTATORS}{det}', check=False)
print("\nDownload audit/human_study/ and distribute everything EXCEPT key.json.")

In [ ]:
# ------------------------------------------------- final table
res = C.load_json(f"{OUT}/metrics/metrics.json", {}).get("results", [])
print(f"{'detector':30s}{'tag':12s}{'n':>6}{'AUC':>8}{'FS':>10}"
      f"{'CR-prior':>10}  verdict")
for r in sorted(res, key=lambda x: (str(x.get('detector')), str(x.get('tag')))):
    v = ("FAITHFUL" if r.get("faithful") else
         "fwd only" if r.get("faithful_forward") else "UNFAITHFUL")
    print(f"{str(r.get('detector'))[:29]:30s}{str(r.get('tag'))[:11]:12s}"
          f"{r.get('n_samples',0):>6}{r.get('AUC',float('nan')):>8.3f}"
          f"{r.get('FS',float('nan')):>10.4f}"
          f"{r.get('CR_minus_prior',float('nan')):>10.3f}  {v}")

try:
    from IPython.display import HTML, display
    display(HTML(open(f"{OUT}/report.html").read()))
except Exception as exc:
    print(f"(inline report unavailable: {exc}); download "
          f"{OUT}/report.html from the Output panel")

In [ ]:
NEXT_STEP = """Download from the Output panel:
  audit/report.html          self-contained, no external files
  audit/results_table.csv    the main results table
  audit/human_study/         distribute WITHOUT key.json
  audit/metrics/*.json       the complete per-metric output

Before reporting, check:
  * the control ordering still holds in section 2 of the report
  * floor p is low for every detector (otherwise the operator is suspect)
  * any detector whose AUC dropped after tuning is reported as a trade-off"""

# ----------------------------------------------------------- WRAP UP
KU.disk_report()
print(C.banner("NEXT STEP"))
print(NEXT_STEP)